# 3장 예제 — 3.3 이상치 처리

- **교재 출처**: manuscript/30_ch3_전처리.md
- **실행 방법**: example 폴더에서  python "3장/<이 파일>" (교재 코드의 data/ 상대경로를 그대로 쓰기 위해 작업 디렉터리는 example)
- **선행 필요**: 이 절의 코드는 앞 절에서 만든 변수를 이어 쓴다. 단독 실행이 필요하면 같은 장의 _장전체_실행본.py 를 쓰거나 아래 절을 먼저 실행하라 — 01_3-1_왜_전처리가_절반인가.py, 02_3-2_결측치_처리.py
- **실행 상태**: 단독 실행 확인됨(이 파일만 돌려도 끝까지 실행된다).

In [ ]:
# [환경 설정] data/ 경로 자동 감지
import os
if not os.path.exists('data') and os.path.exists('../data'):
    os.chdir('..')

In [ ]:
# --- 실행 준비 (교재에서는 앞 절에서 이미 실행한 코드) --------------------
import numpy as np
import pandas as pd
df = pd.read_csv("data/battery_process_data.csv")
df["건조로_1구간_온도"] = df["건조로_1구간_온도"].replace(9999, np.nan)
df = df.fillna(df.median(numeric_only=True))   # 3.7절 clean_data() 규약

### 에러 코드형 이상치의 식별 — describe가 알려주는 신호

In [ ]:
temp = df['건조로_1구간_온도']
print(temp.describe().round(2))

# 교재 실행 결과 ------------------------------------------------------
#   count    1000.00
#   mean      159.44
#   std       697.87
#   min        99.00
#   25%       107.30
#   50%       110.00
#   75%       112.70
#   max      9999.00
#   Name: 건조로_1구간_온도, dtype: float64

In [ ]:
print(temp.sort_values(ascending=False).head(5))

# 교재 실행 결과 ------------------------------------------------------
#   654    9999.0
#   161    9999.0
#   798    9999.0
#   230    9999.0
#   155    9999.0
#   Name: 건조로_1구간_온도, dtype: float64

In [ ]:
df['건조로_1구간_온도'] = df['건조로_1구간_온도'].replace(9999, np.nan)
print("NaN 개수:", df['건조로_1구간_온도'].isna().sum())

med_d = df['건조로_1구간_온도'].median()
df['건조로_1구간_온도'] = df['건조로_1구간_온도'].fillna(med_d)
print("중앙값으로 대체 완료, 대체값:", round(med_d, 2))
print(df['건조로_1구간_온도'].describe().round(2))

# 교재 실행 결과 ------------------------------------------------------
#   NaN 개수: 5
#   중앙값으로 대체 완료, 대체값: 110.0
#   count    1000.00
#   mean      109.99
#   std         3.90
#   min        99.00
#   25%       107.30
#   50%       110.00
#   75%       112.60
#   max       122.40
#   Name: 건조로_1구간_온도, dtype: float64

### IQR 규칙 — 박스플롯의 수학

In [ ]:
q1 = df['프레스_압력'].quantile(0.25)
q3 = df['프레스_압력'].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
print(f"Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}")
print(f"하한={lower:.2f}, 상한={upper:.2f}")

out = df[(df['프레스_압력'] < lower) | (df['프레스_압력'] > upper)]
print("IQR 기준 이상치 개수:", len(out))
print(out[['Lot_ID', '프레스_압력', '불량_여부']].head(10))

# 교재 실행 결과 ------------------------------------------------------
#   Q1=37.27, Q3=42.32, IQR=5.05
#   하한=29.70, 상한=49.90
#   IQR 기준 이상치 개수: 16
#        Lot_ID  프레스_압력  불량_여부
#   76   L25076    29.6      0
#   141  L25141    52.0      0
#   222  L25222    52.0      0
#   319  L25319    52.0      0
#   455  L25455    49.9      0
#   546  L25546    29.2      0
#   566  L25566    28.0      0
#   616  L25616    28.8      0
#   618  L25618    50.9      0
#   636  L25636    28.0      0

### z-score — 표준편차 몇 배나 벗어났는가

In [ ]:
z = (df['프레스_압력'] - df['프레스_압력'].mean()) / df['프레스_압력'].std()
print("|Z|>3 개수:", (z.abs() > 3).sum())
print("Z 최대/최소:", round(z.max(), 2), round(z.min(), 2))

# 교재 실행 결과 ------------------------------------------------------
#   |Z|>3 개수: 3
#   Z 최대/최소: 3.04 -2.99

### 도메인 지식 기반 물리 범위 검사 — 온도는 200℃일 수 없다

In [ ]:
rules = {'믹싱_온도': (10, 50), '건조로_1구간_온도': (80, 160), '프레스_압력': (20, 80)}
for col, (lo, hi) in rules.items():
    n = ((df[col] < lo) | (df[col] > hi)).sum()
    print(f"{col}: 물리 범위 ({lo}~{hi}) 위반 {n}건")

# 교재 실행 결과 ------------------------------------------------------
#   믹싱_온도: 물리 범위 (10~50) 위반 0건
#   건조로_1구간_온도: 물리 범위 (80~160) 위반 0건
#   프레스_압력: 물리 범위 (20~80) 위반 0건